# 학습 정상 분포를 고정한 tail / Gaussian copula 앙상블

https://arxiv.org/html/2009.09463v1
https://pyod.readthedocs.io/en/latest/_modules/pyod/models/copod.html

COPOD의 ECDF/왜도 방향 tail 점수 핵심을 참고하되 평가 배치를 분포 추정에 합치지 않는 frozen inductive 변형. +1 smoothing 적용, 공식 COPOD 재현 아님. 별도 Gaussian copula 후보는 정상 ECDF를 Gaussian rank로 변환하고 LedoitWolf 공분산 거리로 드문 조합을 측정한다. Gaussian 후보는 COPOD 알고리즘이 아닌 별도 실험이다. 두 센서 관점 최대 점수 앙상블.

정상만 분포 학습, 내부 구간에서 임계값/후보 선택. 동일 1325개/10개 관측구간 중첩 검증, 새 독립 검증 아님. RH만 사용. 라벨 부족을 해결했다는 주장이 아님.


In [1]:
from pathlib import Path
import json,copy,numpy as np,pandas as pd,torch
from torch import nn
from sklearn.metrics import confusion_matrix
torch.set_num_threads(1)
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/74_product_normalized_joint_gas_modes.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.rsplit('memo={}',1)[0])
memo={};logs=[]
from scipy.stats import skew,norm
from sklearn.covariance import LedoitWolf
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 allviews=[]
 for v,(tr,te) in enumerate(views(ref,q)):
  normal=tr[y[ref]==0];sort=np.sort(normal,axis=0);n=len(normal)
  asym=np.sign(np.nan_to_num(skew(normal,axis=0)))
  def ranks(x):
   left=np.column_stack([(np.searchsorted(sort[:,j],x[:,j],side='right')+1)/(n+2) for j in range(x.shape[1])])
   right=np.column_stack([(n-np.searchsorted(sort[:,j],x[:,j],side='left')+1)/(n+2) for j in range(x.shape[1])])
   return left,right
  def tail(x):
   l,r=ranks(x);a=-np.log(l);b=-np.log(r)
   sided=np.where(asym>0,b,np.where(asym<0,a,a+b))
   return np.maximum(sided,(a+b)/2).sum(1)
  def gauss(x):
   l,r=ranks(x);u=(l+1-r)/2
   return norm.ppf(np.clip(u,1/(n+2),1-1/(n+2)))
  cov=LedoitWolf().fit(gauss(normal))
  train=np.column_stack([tail(normal),cov.mahalanobis(gauss(normal))])
  out=np.column_stack([tail(te),cov.mahalanobis(gauss(te))])
  allviews.append(out/np.maximum(np.quantile(train,.95,axis=0),1e-6))
  logs.append(dict(train=len(ref),query=len(q),view=v,normal_training=len(normal),gas_training=0,train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 memo[key]=np.max(allviews,axis=0)
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 outer=predict(ref,q);inner=np.full((len(ref),2),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['frozen_COPOD_inspired','Gaussian_copula_joint']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   ip=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],ip,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,outer[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(outer[j,k]>cut),score=float(outer[j,k]),threshold=cut))
 selected=max(candidates,key=lambda a:a[:3]);choices.append(dict(block=int(b),selected=selected[3],budget=selected[4],inner_recall=selected[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(selected[5][j]),score=np.nan,threshold=np.nan))
 print('Finished Copula block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget,product),g in p.groupby(['model','budget','product']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(logs),'training_audit')]:
 obj.to_csv(tables/f'gas_frozen_copula_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))



CN7 extra metadata 713 sensor-consistent unique inputs 713 conflicting 0


Finished Copula block 1


Finished Copula block 2


Finished Copula block 3


Finished Copula block 4


...

Finished Copula block 7


Finished Copula block 8


Finished Copula block 9


Finished Copula block 10


                      model  budget product  TP  FP  FN  TN   recall       F1
      Gaussian_copula_joint    0.02     CN7   2  15   8 680 0.200000 0.148148
      Gaussian_copula_joint    0.02     RG3   1  13  18 588 0.052632 0.060606
      Gaussian_copula_joint    0.05     CN7   4  29   6 666 0.400000 0.186047
      Gaussian_copula_joint    0.05     RG3   1  25  18 576 0.052632 0.044444
      Gaussian_copula_joint    0.10     CN7   6  62   4 633 0.600000 0.153846
      Gaussian_copula_joint    0.10     RG3   2  51  17 550 0.105263 0.055556
      frozen_COPOD_inspired    0.02     CN7   5  23   5 672 0.500000 0.263158
      frozen_COPOD_inspired    0.02     RG3   0   5  19 596 0.000000 0.000000
      frozen_COPOD_inspired    0.05     CN7   6  34   4 661 0.600000 0.240000
      frozen_COPOD_inspired    0.05     RG3   0  12  19 589 0.000000 0.000000
      frozen_COPOD_inspired    0.10     CN7   6  69   4 626 0.600000 0.141176
      frozen_COPOD_inspired    0.10     RG3   1  48  18 553 0.05